In [1]:
# Pass marks, set before the run.
PILOT_LAWS = 50  # the first laws of the 600-law sample every other notebook uses
MAX_COST_USD = 10.0  # stop before the full pilot if the first laws project past this
PROJECTION_LAWS = 5
MAX_MALFORMED_SHARE = 0.05  # replies that are not the JSON asked for
# Worth a full run only if the stronger model gains at least this much exact-label F1 in a method, with a
# paired 95% interval that excludes zero.
WORTH_A_FULL_RUN = 0.05

# 18 · Pilot: a stronger LLM in the best LLM method

Every LLM method so far used small or mid-sized open models through Hugging Face's router. This pilot asks
whether a much stronger model would change the picture, before paying for a full run. It swaps
Claude Opus 5.5, in fast mode, deployed on Microsoft Foundry, into the best LLM method on the EU laws,
**KI-FSPrompt** (notebook 09), and scores it on the same laws as the saved run. Its keyword writers were
Gemma 3 12B and 27B and its rater Llama 3.1 8B. Here Opus writes the keywords and rates the labels, so it makes
two calls a law instead of three. Everything else (prompts, examples, label search) is unchanged.

Two reductions:

- **The cut-off is not re-chosen.** KI-FSPrompt keeps at most 10 labels scoring at least 0.1, the cut chosen
  for the original models on 200 development laws. Choosing it again for Opus would cost another 200 laws.
- **50 laws only,** so the intervals are wide. This decides whether a full run is worth paying for; it is not
  a result to report on its own.

Our first run of this pilot also tried Opus in the hybrid shortlist (notebook 04). Its candidates came from a
trained model that was later retrained, and Omikuji's training cannot be seeded, so the candidates changed and
its saved replies no longer match. It is left out rather than reported without a notebook that reproduces it.

Opus 5.5 takes no temperature, so its replies can differ between runs; the saved replies are reused on a
rerun. Its thinking is always on, at the model's default effort, and is paid for as output.

In [2]:
from dataclasses import replace

import pandas as pd

from tiered_taxonomy import experiment as ex
from tiered_taxonomy import runs
from tiered_taxonomy import signals as sg
from tiered_taxonomy.config import data_root, load_config
from tiered_taxonomy.llm import Client, ReplyStore
from tiered_taxonomy.methods.trained import Cut
from tiered_taxonomy.metrics import micro_f1, paired_difference
from tiered_taxonomy.models import spec_from_config

root = data_root()
config = load_config()
seed = config["seed"]
settings = config["samples"]["eurlex"]
eurlex = ex.load_eurlex(root)
full = ex.sample_eurlex(
    eurlex, settings["normal_documents"], settings["long_documents"], settings["long_threshold_words"], seed
)
sample = full[:PILOT_LAWS]
ids = [item.id for item in sample]
gold = ex.gold_exact(sample)
strong = spec_from_config(config["models"]["strong"])
print(strong.provider, strong.model, f"${strong.input_price} / ${strong.output_price} per million tokens")
print(len(sample), "laws")

/Users/matthewharris/git/tiered-taxonomy-tagging/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


anthropic_foundry claude-opus-5-5-fast $8.0 / $40.0 per million tokens
50 laws


## KI-FSPrompt, with Opus in place of the original models

The first laws run first, and the pilot stops if they project past the cost limit.

In [3]:
bench = runs.make_bench(eurlex, root, config)
replies = ReplyStore(root / "replies.sqlite")
runs.load_cards(bench)
runs.embed_inputs(bench, sample)
tools = replace(
    runs.make_recipe_tools(bench), generators=[Client(strong, replies)], ranker=Client(strong, replies)
)
paper = ex.load_results("09_paper_recipes")["eurlex"]["ki_fsprompt"]
ki_cut = Cut(paper["limit"], paper["threshold"])


def run_ki(items):
    ranked, record = runs.run_recipe(bench, tools, "ki_fsprompt", "test", items)
    return [ki_cut.apply(ranking) for ranking in ranked], record


ARMS = {"ki_fsprompt": run_ki}
first = {name: run(sample[:PROJECTION_LAWS])[1] for name, run in ARMS.items()}
projected = sum(record.cost_usd / record.items for record in first.values()) * PILOT_LAWS
print(f"projected cost for {PILOT_LAWS} laws: ${projected:.2f}")
if projected > MAX_COST_USD:
    raise RuntimeError(f"the pilot would cost about ${projected:.2f}, over the ${MAX_COST_USD:.2f} limit")

projected cost for 50 laws: $3.46


In [4]:
labels, records = {}, {}
for name, run in ARMS.items():
    labels[name], records[name] = run(sample)
    ex.save_predictions(root, eurlex.name, f"{name}_opus_fast", ids, labels[name])
    share = records[name].malformed / records[name].calls
    print(name, records[name].__dict__, f"malformed {share:.1%}")
    if share > MAX_MALFORMED_SHARE:
        raise RuntimeError(f"{name}: {share:.1%} of replies were malformed, over {MAX_MALFORMED_SHARE:.0%}")

ki_fsprompt {'method': 'ki_fsprompt', 'items': 50, 'seconds': 1.1, 'calls': 100, 'malformed': 0, 'invalid': 0, 'cost_usd': 3.4077} malformed 0.0%


## Against the saved run on the same laws

The saved run's labels are read back for the same 50 laws. Fine-tuned Legal-BERT, trained with the newer
laws and cut as in notebook 14, is shown for reference.

In [5]:
like = ex.load_results("14_like_for_like")["eurlex"]["ranked"]["classifier:newer"]
bert_rankings = sg.load_sources(root, sg.NEWER_LAWS_SUFFIX.format("eurlex"), ["classifier"], "test")
bert = [Cut(like["limit"], like["threshold"]).apply(bert_rankings["classifier"][i]) for i in ids]
ORIGINAL = {"ki_fsprompt": "Gemma 3 12B and 27B, Llama 3.1 8B"}

results = {"laws": len(sample), "model": strong.model, "provider": strong.provider, "methods": {}}
rows = {}
for name in ARMS:
    saved = ex.load_predictions(root, eurlex.name, name, ids)
    before, after = ex.evaluate(eurlex, sample, saved), ex.evaluate(eurlex, sample, labels[name])
    gain = paired_difference(micro_f1, gold, labels[name], saved)
    results["methods"][name] = {
        "original_models": ORIGINAL[name],
        "original": before,
        "opus_fast": after,
        "gain": [round(value, 4) for value in gain],
        "run": records[name].__dict__,
        "cost_per_1000": round(records[name].cost_usd / records[name].items * 1000, 2),
    }
    for who, scores in ((ORIGINAL[name], before), ("Claude Opus 5.5 fast", after)):
        rows[f"{name}: {who}"] = {
            "exact F1": scores["exact"]["f1"],
            "precision": scores["exact"]["precision"],
            "recall": scores["exact"]["recall"],
            **{f"tier {t}": s["f1"] for t, s in scores["tiers"].items()},
            "labels per law": scores["labels_per_item"],
        }
results["fine_tuned_legal_bert"] = ex.evaluate(eurlex, sample, bert)
rows["reference: fine-tuned Legal-BERT, with newer laws"] = {
    "exact F1": results["fine_tuned_legal_bert"]["exact"]["f1"],
    "precision": results["fine_tuned_legal_bert"]["exact"]["precision"],
    "recall": results["fine_tuned_legal_bert"]["exact"]["recall"],
    **{f"tier {t}": s["f1"] for t, s in results["fine_tuned_legal_bert"]["tiers"].items()},
    "labels per law": results["fine_tuned_legal_bert"]["labels_per_item"],
}
pd.DataFrame(rows).T.round(3)

,exact F1,precision,recall,tier 1,tier 2,tier 3,labels per law
"ki_fsprompt: Gemma 3 12B and 27B, Llama 3.1 8B",0.342,0.281,0.439,0.682,0.575,0.532,9.76
ki_fsprompt: Claude Opus 5.5 fast,0.463,0.376,0.603,0.769,0.698,0.696,10.00
"reference: fine-tuned Legal-BERT, with newer laws",0.530,0.591,0.481,0.770,0.721,0.716,5.08


In [6]:
verdict = {}
for name, entry in results["methods"].items():
    point, low, high = entry["gain"]
    verdict[name] = {
        "exact F1 gain": point,
        "95% interval": (low, high),
        "$ per 1,000 laws": entry["cost_per_1000"],
        "worth a full run": point >= WORTH_A_FULL_RUN and low > 0,
    }
results["verdict"] = verdict
pd.DataFrame(verdict).T

,exact F1 gain,95% interval,"$ per 1,000 laws",worth a full run
ki_fsprompt,0.1206,"(0.0852, 0.1558)",68.15,True


In [7]:
ex.save_results("18_stronger_llm_pilot", results)
print("saved")

saved
